# VigieMer — 01 · Première détection de navires dans le golfe de Guinée

**Objectif de ce notebook :** prendre de vraies images radar Sentinel-1 récentes au large du Togo, du Bénin et du Nigeria, et y détecter automatiquement tous les navires avec un modèle d'IA.

**Avant de lancer :** dans le panneau de droite → *Session options*
- **Accelerator : GPU T4 x2**
- **Internet : On**

Puis *Run All*. Durée attendue : 10 à 20 minutes (surtout le téléchargement des images).

Sources : images Sentinel-1 (ESA / Copernicus) servies par Microsoft Planetary Computer, modèle de détection open source d'Allen AI (Apache-2.0).

## 1. Réglages

In [ ]:
# Zone surveillée : rectangle [lon_min, lat_min, lon_max, lat_max]
# Ici : de Lomé (Togo) à l'est de Lagos (Nigeria), côte et large
BBOX = [0.8, 4.8, 4.2, 6.6]

# On cherche les images des N derniers jours...
DAYS_BACK = 30
# ...en ignorant les plus récentes. Global Fishing Watch publie l'AIS avec ~4 jours
# de retard : pour pouvoir croiser avec l'AIS (notebook 02), on en laisse 4.
END_DAYS_AGO = 4

# Nombre d'images à analyser (traitées 2 par 2, une par GPU : ~10 min par paire)
N_SCENES = 6

# Points de mer qui nous intéressent : une image n'est retenue que si elle couvre
# au moins l'un d'eux (sinon on risque une image presque entièrement sur terre)
SEA_POINTS = {"Rade de Lagos": (3.40, 6.25), "Large de Cotonou": (2.40, 6.15),
              "Large de Lomé": (1.25, 6.00)}

# Seuil de confiance du détecteur (0.9 = réglage d'Allen AI, peu de fausses alertes)
CONF = 0.9

# Dossier de sortie (tout ce qui est dans /kaggle/working est conservé)
OUT_ROOT = "/kaggle/working/vigiemer_outputs"

## 2. Vérifier les deux GPU

In [ ]:
import torch                                   # bibliothèque de deep learning
n_gpu = torch.cuda.device_count()              # nombre de GPU visibles
print("GPU disponibles :", n_gpu)
for i in range(n_gpu):                         # description de chaque GPU
    p = torch.cuda.get_device_properties(i)
    print(f"  cuda:{i} -> {p.name}, {p.total_memory / 1e9:.1f} Go")
# Sans GPU tout marche quand même, mais environ 10 fois plus lentement
if n_gpu == 0:
    print("ATTENTION : aucun GPU. Active 'GPU T4 x2' dans Session options.")
N_WORKERS = max(n_gpu, 1)                      # un processus par GPU

## 3. Installer les bibliothèques manquantes

In [ ]:
# pystac-client / planetary-computer : chercher et télécharger les images satellite
# rasterio : lire et reprojeter les images radar
# global-land-mask : savoir si un point est sur terre ou en mer
# folium : la carte interactive
!pip install -q pystac-client planetary-computer rasterio global-land-mask folium shapely

## 4. Récupérer le modèle d'Allen AI (code + poids pré-entraînés)

In [ ]:
import os, subprocess, urllib.request

REPO = "/kaggle/working/vessel-detection-sentinels"
# On clone le dépôt officiel (code des modèles)
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "-q", "--depth", "1",
                    "https://github.com/allenai/vessel-detection-sentinels.git", REPO], check=True)

# Les poids sont stockés avec Git LFS : le clone ne contient que des "pointeurs".
# On télécharge les vrais fichiers (122 Mo et 96 Mo).
BASE = "https://media.githubusercontent.com/media/allenai/vessel-detection-sentinels/main/"
WEIGHTS = ["data/model_artifacts/sentinel-1/frcnn_cmp2/3dff445/best.pth",
           "data/model_artifacts/sentinel-1/attr/c34aa37/best.pth"]
for rel in WEIGHTS:
    path = os.path.join(REPO, rel)
    if os.path.getsize(path) < 1_000_000:          # pointeur LFS = quelques octets
        urllib.request.urlretrieve(BASE + rel, path)
    print(f"{rel.split('/')[-3]:>12} : {os.path.getsize(path) / 1e6:.0f} Mo")

## 5. Le moteur de traitement

Ce fichier est écrit sur le disque pour que chaque GPU puisse le lancer dans son propre processus (c'est ce qui permet le travail en parallèle).

In [ ]:
%%writefile /kaggle/working/vigiemer_worker.py
# =====================================================================
# vigiemer_worker.py — le "moteur" de VigieMer
# Ce fichier est exécuté dans des processus séparés, un par GPU.
# Chaque processus traite UNE image satellite de bout en bout :
#   téléchargement -> reprojection -> détection IA -> fiche des navires
# =====================================================================

import os            # gestion des fichiers et dossiers
import sys           # pour modifier le chemin d'import Python
import time          # pour chronométrer chaque étape
import math          # pour pi (taille de la Terre en Web Mercator)
import traceback     # pour renvoyer une erreur lisible au notebook
import warnings      # pour couper les avertissements inoffensifs de pandas

# Le code d'Allen AI déclenche des centaines d'avertissements pandas sans gravité
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np   # calcul sur les tableaux de pixels

# Dossier où le notebook a cloné le code d'Allen AI
REPO = "/kaggle/working/vessel-detection-sentinels"

# Les deux modèles pré-entraînés fournis par Allen AI (licence Apache-2.0)
MODELS = {
    # le détecteur : il trouve les navires (Faster R-CNN)
    "detector": os.path.join(REPO, "data/model_artifacts/sentinel-1/frcnn_cmp2/3dff445"),
    # le post-processeur : il estime longueur, largeur, vitesse, type pêche
    "attr": os.path.join(REPO, "data/model_artifacts/sentinel-1/attr/c34aa37"),
}

# Circonférence de la Terre en projection Web Mercator (EPSG:3857), en mètres
WEB_MERCATOR_M = 2 * math.pi * 6378137
# Taille du pixel utilisée par Allen AI pour Sentinel-1 : niveau de zoom 13,
# soit ~9,55 m, juste sous la résolution native de 10 m. On garde la même
# valeur pour donner au modèle exactement le type d'image vu à l'entraînement.
PIXEL_M = WEB_MERCATOR_M / 512 / (2 ** 13)


def _prepare_imports():
    """Rend le code d'Allen AI importable sur Kaggle."""
    # On ajoute le dépôt et son dossier src au chemin d'import
    for p in (REPO, os.path.join(REPO, "src")):
        if p not in sys.path:
            sys.path.insert(0, p)
    # Le code d'Allen AI importe GDAL (osgeo), lourd à installer sur Kaggle.
    # Nous n'utilisons AUCUNE fonction GDAL (on fait la reprojection avec
    # rasterio), donc on met un "faux" module à la place s'il est absent.
    try:
        from osgeo import gdal  # noqa: F401
    except ImportError:
        from unittest import mock
        fake = mock.MagicMock()
        sys.modules["osgeo"] = fake
        sys.modules["osgeo.gdal"] = fake.gdal
    # Depuis PyTorch 2.6, torch.load refuse par défaut certains fichiers.
    # Les poids viennent du dépôt officiel d'Allen AI : on les charge normalement.
    import torch
    if not getattr(torch.load, "_vigiemer", False):
        original_load = torch.load

        def patched_load(*args, **kwargs):
            kwargs.setdefault("weights_only", False)
            return original_load(*args, **kwargs)

        patched_load._vigiemer = True
        torch.load = patched_load


def download(url, dst):
    """Télécharge un fichier en flux (les bandes radar font ~1 Go)."""
    import requests
    # Si le fichier est déjà là (relance du notebook), on ne retélécharge pas
    if os.path.exists(dst) and os.path.getsize(dst) > 0:
        return dst
    tmp = dst + ".part"  # fichier temporaire tant que ce n'est pas fini
    with requests.get(url, stream=True, timeout=300) as r:
        r.raise_for_status()                     # erreur claire si le lien échoue
        with open(tmp, "wb") as f:
            for chunk in r.iter_content(1 << 22):  # morceaux de 4 Mo
                f.write(chunk)
    os.replace(tmp, dst)  # on renomme seulement quand tout est reçu
    return dst


CDSE_TOKEN_URL = "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token"
CDSE_CATALOGUE = "https://catalogue.dataspace.copernicus.eu/odata/v1"
CDSE_ZIPPER = "https://zipper.dataspace.copernicus.eu/odata/v1"


def cdse_token():
    """Jeton d'accès Copernicus (valable ~10 min), à partir des identifiants en variables d'env."""
    import requests
    r = requests.post(CDSE_TOKEN_URL, timeout=60, data={
        "client_id": "cdse-public", "grant_type": "password",
        "username": os.environ["CDSE_USER"], "password": os.environ["CDSE_PASSWORD"]})
    r.raise_for_status()
    return r.json()["access_token"]


def cdse_stream(url, dst, token):
    """Télécharge un fichier Copernicus en suivant les redirections À LA MAIN.

    requests supprime l'en-tête d'authentification quand il suit une redirection
    vers un autre serveur ; on le remet donc nous-mêmes à chaque étape.
    """
    import requests
    if os.path.exists(dst) and os.path.getsize(dst) > 0:
        return dst
    session = requests.Session()
    session.headers["Authorization"] = f"Bearer {token}"
    r = session.get(url, allow_redirects=False, stream=True, timeout=300)
    while r.status_code in (301, 302, 303, 307, 308):
        r = session.get(r.headers["Location"], allow_redirects=False, stream=True, timeout=300)
    r.raise_for_status()
    tmp = dst + ".part"
    with open(tmp, "wb") as f:
        for chunk in r.iter_content(1 << 22):
            f.write(chunk)
    os.replace(tmp, dst)
    return dst


def download_cdse(scene, out_dir):
    """Récupère seulement les deux bandes VH et VV d'un produit Copernicus.

    Méthode 1 : l'API "Nodes" permet de prendre deux fichiers dans le produit (~1,6 Go)
    au lieu du produit entier. Méthode 2 (secours) : produit complet en zip, puis extraction.
    """
    import requests, zipfile
    token = cdse_token()
    pid, name = scene["cdse_id"], scene["name"]
    try:
        listing = requests.get(
            f"{CDSE_CATALOGUE}/Products({pid})/Nodes({name})/Nodes(measurement)/Nodes",
            headers={"Authorization": f"Bearer {token}"}, timeout=120)
        listing.raise_for_status()
        files = [n["Name"] for n in listing.json().get("result", [])]
        paths = {}
        for pol in ("vh", "vv"):
            fname = next(f for f in files if f"-{pol}-" in f and f.endswith(".tiff"))
            url = f"{CDSE_ZIPPER}/Products({pid})/Nodes({name})/Nodes(measurement)/Nodes({fname})/$value"
            paths[pol] = cdse_stream(url, os.path.join(out_dir, f"{pol}.tif"), token)
        return paths["vh"], paths["vv"]
    except Exception as e:
        print(f"Nodes indisponible ({str(e)[:80]}), téléchargement du produit complet…")
    zpath = cdse_stream(f"{CDSE_ZIPPER}/Products({pid})/$value",
                        os.path.join(out_dir, "product.zip"), cdse_token())
    out = {}
    with zipfile.ZipFile(zpath) as z:
        for member in z.namelist():
            for pol in ("vh", "vv"):
                if "/measurement/" in member and f"-{pol}-" in member and member.endswith(".tiff"):
                    dst = os.path.join(out_dir, f"{pol}.tif")
                    with z.open(member) as src, open(dst, "wb") as f:
                        while chunk := src.read(1 << 22):
                            f.write(chunk)
                    out[pol] = dst
    os.remove(zpath)
    return out["vh"], out["vv"]


def warp_pair(vh_path, vv_path):
    """Reprojette les deux polarisations (VH, VV) sur la même grille EPSG:3857.

    Une image Sentinel-1 GRD brute n'est pas "à plat" sur une carte : elle est
    dans la géométrie du satellite, avec des points de calage (GCP). On la
    redresse en Web Mercator, comme Allen AI l'a fait pour entraîner le modèle.
    """
    import rasterio
    from rasterio.crs import CRS
    from rasterio.warp import calculate_default_transform, reproject, Resampling

    dst_crs = CRS.from_epsg(3857)   # projection de destination
    bands, transform, shape = [], None, None
    for path in (vh_path, vv_path):  # ordre VH puis VV : celui attendu par le modèle
        with rasterio.open(path) as src:
            gcps, gcp_crs = src.gcps          # points de calage de l'image
            data = src.read(1)                # valeurs radar brutes (uint16)
            if gcps:
                src_kw = dict(gcps=gcps, src_crs=gcp_crs)
            else:  # au cas où l'image serait déjà géoréférencée
                src_kw = dict(src_transform=src.transform, src_crs=src.crs)
            if transform is None:
                # La grille est calculée une seule fois (sur VH) puis réutilisée
                # pour VV : les deux bandes sont ainsi parfaitement alignées.
                if gcps:
                    transform, w, h = calculate_default_transform(
                        gcp_crs, dst_crs, src.width, src.height,
                        gcps=gcps, resolution=PIXEL_M)
                else:
                    transform, w, h = calculate_default_transform(
                        src.crs, dst_crs, src.width, src.height,
                        *src.bounds, resolution=PIXEL_M)
                shape = (h, w)
            dst = np.zeros(shape, np.uint16)  # image redressée, vide au départ
            reproject(source=data, destination=dst,
                      dst_transform=transform, dst_crs=dst_crs,
                      resampling=Resampling.bilinear,   # même méthode qu'Allen AI
                      src_nodata=0, dst_nodata=0,
                      num_threads=4, **src_kw)
            del data
            # Allen AI coupe les valeurs à 255 et passe en uint8 : on fait pareil
            bands.append(np.clip(dst, 0, 255).astype(np.uint8))
            del dst
    return np.stack(bands), transform  # tableau (2, H, W) + géoréférencement


class PixelToLonLat:
    """Convertit (colonne, ligne) de l'image en (longitude, latitude).

    Le code d'Allen AI attend un objet GDAL avec une méthode TransformPoint.
    On fournit la même méthode, calculée avec rasterio + pyproj.
    """

    def __init__(self, transform):
        from pyproj import Transformer
        self.transform = transform  # pixel -> mètres Web Mercator
        self.to_wgs84 = Transformer.from_crs(3857, 4326, always_xy=True)

    def TransformPoint(self, direction, col, row, z=0):
        x, y = self.transform * (col, row)          # pixel -> mètres
        lon, lat = self.to_wgs84.transform(x, y)     # mètres -> degrés
        return 1, (lon, lat, 0.0)                    # 1 = succès (convention GDAL)


def run_scene(scene, gpu_id, out_root, conf=0.9, keep_raw=False):
    """Traite une scène complète sur le GPU numéro gpu_id."""
    log, t0 = [], time.time()

    def step(msg):  # petit journal horodaté renvoyé au notebook
        log.append(f"[GPU {gpu_id}] {time.time() - t0:6.0f}s  {msg}")

    try:
        _prepare_imports()
        import torch
        import pandas as pd
        from PIL import Image
        from src.inference.pipeline import apply_model

        # Chaque processus travaille sur SON GPU
        if torch.cuda.is_available():
            device = torch.device(f"cuda:{gpu_id}")
            torch.cuda.set_device(device)
        else:
            device = torch.device("cpu")
        step(f"appareil : {device}")

        sid = scene["id"]
        out_dir = os.path.join(out_root, sid)
        os.makedirs(out_dir, exist_ok=True)

        # 1) Téléchargement des deux polarisations
        if scene.get("source") == "cdse":           # source officielle Copernicus
            vh, vv = download_cdse(scene, out_dir)
        else:                                        # Microsoft Planetary Computer
            vh = download(scene["vh"], os.path.join(out_dir, "vh.tif"))
            vv = download(scene["vv"], os.path.join(out_dir, "vv.tif"))
        step("téléchargement terminé")

        # 2) Redressement de l'image
        img, transform = warp_pair(vh, vv)
        step(f"image redressée : {img.shape[2]} x {img.shape[1]} pixels")

        # 3) Détection IA (fenêtres glissantes de 2048 px, réglages d'Allen AI)
        pred = apply_model(
            MODELS["detector"], torch.as_tensor(img),
            window_size=2048, padding=400, overlap=20,
            threshold=conf,                       # confiance minimale
            transformer=PixelToLonLat(transform),
            nms_thresh=10,                        # fusionne les doublons proches
            postprocess_model_dir=MODELS["attr"],
            out_path=out_dir, catalog="sentinel1", device=device,
        )
        step(f"{len(pred)} navires détectés")

        # 4) On enrichit la fiche de chaque navire
        pred = pred.reset_index(drop=True)
        pred["scene_id"] = sid
        pred["acquired"] = scene["datetime"]
        # Le cap est donné en 16 secteurs de 22,5° : on garde le plus probable
        heading_cols = [f"heading_bucket_{i}" for i in range(16)]
        if len(pred):
            pred["heading_deg"] = pred[heading_cols].values.argmax(axis=1) * 22.5
        else:
            pred["heading_deg"] = []
        pred = pred.drop(columns=heading_cols)
        # Détection sur terre ? (le modèle est fait pour la mer : on signale)
        try:
            from global_land_mask import globe
            pred["on_land"] = [bool(globe.is_land(la, lo)) for la, lo in zip(pred.lat, pred.lon)]
        except Exception:
            pred["on_land"] = False

        # 5) Vignettes 80x80 px (≈ 760 m de côté) pour CHAQUE détection (max 800)
        #    Contraste étiré par vignette (percentiles 2-99,5) pour bien voir la coque.
        crops_dir = os.path.join(out_dir, "crops")
        os.makedirs(crops_dir, exist_ok=True)
        pred["crop"] = ""
        H, W = img.shape[1], img.shape[2]
        for i, r in pred.sort_values("score", ascending=False).head(800).iterrows():
            rr = int(np.clip(r.preprocess_row, 40, H - 40))
            cc = int(np.clip(r.preprocess_column, 40, W - 40))
            patch = img[1, rr - 40:rr + 40, cc - 40:cc + 40].astype(np.float32)  # bande VV
            lo, hi = np.percentile(patch, 2), np.percentile(patch, 99.5)
            patch = np.clip((patch - lo) / max(hi - lo, 1) * 255, 0, 255).astype(np.uint8)
            name = f"det_{i:04d}.png"
            Image.fromarray(patch).save(os.path.join(crops_dir, name))
            pred.loc[i, "crop"] = f"{sid}/crops/{name}"   # chemin relatif, valable partout

        # Fiche de la scène (emprise au sol, heure, source) pour la carte de l'application
        import json
        meta = {k: v for k, v in scene.items() if k in ("id", "datetime", "source", "footprint")}
        corners = [PixelToLonLat(transform).TransformPoint(0, c, r)[1][:2]
                   for c, r in ((0, 0), (W, 0), (W, H), (0, H))]
        meta["bounds"] = corners
        meta["n_detections"] = int(len(pred))
        with open(os.path.join(out_dir, "scene_meta.json"), "w") as f:
            json.dump(meta, f)

        # 6) Aperçu réduit de toute la scène (1 pixel sur 16) pour les figures
        np.save(os.path.join(out_dir, "quicklook_vv.npy"), img[1, ::16, ::16])
        csv_path = os.path.join(out_dir, "detections.csv")
        pred.to_csv(csv_path, index=False)
        step("fiches et vignettes enregistrées")

        if not keep_raw:  # on libère le disque Kaggle (≈ 2 Go par scène)
            for p in (vh, vv):
                os.remove(p)

        return {"id": sid, "ok": True, "csv": csv_path, "out_dir": out_dir,
                "n": len(pred), "seconds": round(time.time() - t0), "log": log}
    except Exception:
        return {"id": scene.get("id"), "ok": False,
                "error": traceback.format_exc(), "log": log}


def run_on_gpu(args):
    """Petit adaptateur pour ProcessPoolExecutor.map (un seul argument)."""
    return run_scene(*args)

## 6. Chercher les images Sentinel-1 de la zone

Deux sources possibles, essayées dans l'ordre :
1. **Microsoft Planetary Computer** : public, sans compte. Mais il peut refuser temporairement les machines Kaggle (« rate limited »). On réessaie alors plusieurs fois en patientant.
2. **Copernicus Data Space** (ESA) : la source officielle, qui demande un compte gratuit. Elle sert de secours si Microsoft refuse toujours. Il faut alors deux secrets Kaggle : `CDSE_USER` (ton email Copernicus) et `CDSE_PASSWORD`.

In [ ]:
import os, time
import datetime as dt
import pandas as pd
import requests

end = dt.datetime.now(dt.timezone.utc) - dt.timedelta(days=END_DAYS_AGO)
start = end - dt.timedelta(days=DAYS_BACK)

def search_planetary(retries=5):
    """Source 1 : Microsoft Planetary Computer, avec relances si le serveur refuse."""
    import pystac_client, planetary_computer
    wait = 30                                        # première pause : 30 s
    for attempt in range(1, retries + 1):
        try:
            catalog = pystac_client.Client.open(
                "https://planetarycomputer.microsoft.com/api/stac/v1",
                modifier=planetary_computer.sign_inplace)   # ajoute un jeton de téléchargement
            search = catalog.search(
                collections=["sentinel-1-grd"],          # images radar "brutes" (GRD)
                bbox=BBOX,
                datetime=f"{start:%Y-%m-%d}/{end:%Y-%m-%d}",
                query={"sar:instrument_mode": {"eq": "IW"}})
            scenes = []
            for it in search.items():
                if "vh" in it.assets and "vv" in it.assets:   # il faut les deux polarisations
                    scenes.append({"id": it.id, "datetime": it.datetime.isoformat(),
                                   "source": "planetary", "footprint": it.geometry,
                                   "vh": it.assets["vh"].href, "vv": it.assets["vv"].href})
            return scenes
        except Exception as e:
            print(f"  Planetary Computer, essai {attempt}/{retries} refusé : {str(e)[:120]}")
            if attempt < retries:
                time.sleep(wait)
                wait *= 2                                # 30 s, 60 s, 120 s, 240 s...
    return None

def search_copernicus():
    """Source 2 : catalogue officiel Copernicus (la recherche ne demande pas de compte)."""
    lon0, lat0, lon1, lat1 = BBOX
    poly = f"POLYGON(({lon0} {lat0},{lon1} {lat0},{lon1} {lat1},{lon0} {lat1},{lon0} {lat0}))"
    filt = (f"Collection/Name eq 'SENTINEL-1' "
            f"and contains(Name,'IW_GRDH_1SDV') and not contains(Name,'_COG') "
            f"and OData.CSC.Intersects(area=geography'SRID=4326;{poly}') "
            f"and ContentDate/Start gt {start:%Y-%m-%dT%H:%M:%S.000Z} "
            f"and ContentDate/Start lt {end:%Y-%m-%dT%H:%M:%S.000Z}")
    r = requests.get("https://catalogue.dataspace.copernicus.eu/odata/v1/Products",
                     params={"$filter": filt, "$orderby": "ContentDate/Start desc", "$top": 50},
                     timeout=120)
    r.raise_for_status()
    scenes = []
    for p in r.json().get("value", []):
        scenes.append({"id": p["Name"].replace(".SAFE", ""),
                       "datetime": p["ContentDate"]["Start"],
                       "footprint": p.get("GeoFootprint"),
                       "source": "cdse", "cdse_id": p["Id"], "name": p["Name"]})
    return scenes

scenes = search_planetary()
if not scenes:
    print("Bascule sur Copernicus Data Space…")
    # Les identifiants sont lus dans les secrets Kaggle et passés aux processus GPU
    # par variables d'environnement (jamais affichés).
    try:
        from kaggle_secrets import UserSecretsClient
        sec = UserSecretsClient()
        os.environ["CDSE_USER"] = sec.get_secret("CDSE_USER")
        os.environ["CDSE_PASSWORD"] = sec.get_secret("CDSE_PASSWORD")
    except Exception:
        raise SystemExit("Planetary Computer refuse et les secrets CDSE_USER / CDSE_PASSWORD "
                         "sont absents. Ajoute-les (Add-ons -> Secrets) ou relance plus tard.")
    scenes = search_copernicus()

from shapely.geometry import shape, Point

def points_covered(scene):
    # Liste des points de mer (SEA_POINTS) couverts par l'emprise de l'image
    try:
        geom = shape(scene["footprint"])
    except Exception:
        return []
    return [name for name, (lo, la) in SEA_POINTS.items() if geom.contains(Point(lo, la))]

for sc in scenes:
    sc["covers"] = points_covered(sc)
scenes.sort(key=lambda s: s["datetime"], reverse=True)      # plus récentes d'abord

# Sélection : images couvrant un point de mer, en privilégiant des passages
# (jours / heures) différents, puis la rade de Lagos, la plus fréquentée
selected, seen_passes = [], set()
candidates = sorted([s for s in scenes if s["covers"]], key=lambda s: s["datetime"], reverse=True)
candidates.sort(key=lambda s: "Rade de Lagos" not in s["covers"])   # tri stable : Lagos d'abord
for sc in candidates:
    pass_key = sc["datetime"][:13]                  # même passage = même heure
    if pass_key in seen_passes:
        continue
    selected.append(sc); seen_passes.add(pass_key)
    if len(selected) == N_SCENES:
        break
if len(selected) < N_SCENES:                         # complète si besoin
    for sc in scenes:
        if sc not in selected and sc["covers"]:
            selected.append(sc)
        if len(selected) == N_SCENES:
            break
selected.sort(key=lambda s: s["datetime"], reverse=True)

table = pd.DataFrame([{"id": s["id"][:45], "date (UTC)": s["datetime"][:16].replace("T", " "),
                       "source": s["source"], "couvre": ", ".join(s["covers"]),
                       "retenue": s in selected} for s in scenes])
print(f"{len(scenes)} images trouvées entre le {start:%d/%m} et le {end:%d/%m}, {len(selected)} retenues")
table

## 7. Lancer la détection — une image par GPU, en parallèle

In [ ]:
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor
import sys
sys.path.insert(0, "/kaggle/working")            # pour trouver vigiemer_worker.py
import vigiemer_worker

assert selected, "Aucune image retenue : augmente DAYS_BACK ou vérifie SEA_POINTS."

results = []
# On traite les images par paires : une par GPU, les deux en même temps
for k in range(0, len(selected), N_WORKERS):
    batch = selected[k:k + N_WORKERS]
    jobs = [(scene, gpu_id, OUT_ROOT, CONF) for gpu_id, scene in enumerate(batch)]
    # Les liens Microsoft expirent au bout d'environ une heure : on les renouvelle à chaque paire
    for sc in batch:
        if sc["source"] == "planetary":
            import planetary_computer
            for pol in ("vh", "vv"):
                sc[pol] = planetary_computer.sign(sc[pol].split("?")[0])
    print(f"Paire {k // N_WORKERS + 1} / {-(-len(selected) // N_WORKERS)} …")
    # "spawn" : chaque processus démarre proprement avec son propre GPU
    with ProcessPoolExecutor(max_workers=N_WORKERS, mp_context=mp.get_context("spawn")) as ex:
        results += list(ex.map(vigiemer_worker.run_on_gpu, jobs))

for res in results:                               # compte rendu de chaque GPU
    print("\n".join(res["log"]))
    if res["ok"]:
        print(f"OK  {res['id']} : {res['n']} navires en {res['seconds']} s\n")
    else:
        print(f"ÉCHEC {res['id']} :\n{res['error']}")

## 8. Les navires détectés

In [ ]:
# On rassemble les fiches de toutes les scènes réussies
dets = pd.concat([pd.read_csv(r["csv"]) for r in results if r["ok"]], ignore_index=True)
sea = dets[~dets.on_land]                         # on écarte ce qui tombe sur terre
print(f"Total : {len(dets)} détections, dont {len(sea)} en mer")

cols = ["acquired", "lat", "lon", "score", "vessel_length_m", "vessel_width_m",
        "vessel_speed_k", "heading_deg", "is_fishing_vessel"]
sea.sort_values("score", ascending=False)[cols].head(20).round(3)

## 9. Vue d'ensemble de chaque scène

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

ok = [r for r in results if r["ok"]]
ncol = min(3, len(ok)); nrow = -(-len(ok) // ncol)
fig, axes = plt.subplots(nrow, ncol, figsize=(7 * ncol, 7 * nrow), squeeze=False)
for ax in axes.ravel()[len(ok):]:
    ax.axis("off")
for ax, r in zip(axes.ravel(), ok):
    ql = np.load(os.path.join(r["out_dir"], "quicklook_vv.npy"))   # image réduite /16
    ax.imshow(ql, cmap="gray", vmin=0, vmax=np.percentile(ql[ql > 0], 99) if (ql > 0).any() else 255)
    d = dets[dets.scene_id == r["id"]]
    m = d[~d.on_land]
    # Les coordonnées pixel sont divisées par 16 comme l'aperçu
    ax.scatter(m.preprocess_column / 16, m.preprocess_row / 16, s=30,
               facecolors="none", edgecolors="#ff3b30", linewidths=1.2)
    ax.set_title(f"{r['id'][:32]}…\n{len(m)} navires en mer", fontsize=11)
    ax.axis("off")
plt.tight_layout()
plt.savefig(os.path.join(OUT_ROOT, "overview.png"), dpi=120)
plt.show()

## 10. Zoom sur les détections les plus sûres

In [ ]:
from PIL import Image
top = sea[sea.crop.notna() & (sea.crop != "")].sort_values("score", ascending=False).head(12)
fig, axes = plt.subplots(2, 6, figsize=(18, 6.5))
for ax, (_, r) in zip(axes.ravel(), top.iterrows()):
    ax.imshow(Image.open(os.path.join(OUT_ROOT, r.crop)), cmap="gray")
    ax.set_title(f"{r.vessel_length_m:.0f} m · {r.score:.2f}", fontsize=10)
    ax.axis("off")
for ax in axes.ravel()[len(top):]:
    ax.axis("off")
plt.suptitle("Navires détectés (longueur estimée · confiance)", fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(OUT_ROOT, "crops.png"), dpi=120)
plt.show()

## 11. Carte interactive

In [ ]:
import folium

center = [(BBOX[1] + BBOX[3]) / 2, (BBOX[0] + BBOX[2]) / 2]
fmap = folium.Map(location=center, zoom_start=8, tiles="OpenStreetMap")
folium.Rectangle([[BBOX[1], BBOX[0]], [BBOX[3], BBOX[2]]],
                 color="#8e8e93", weight=1, fill=False).add_to(fmap)
for _, r in sea.iterrows():
    popup = (f"<b>Navire détecté</b><br>{r.acquired[:16]} UTC<br>"
             f"Longueur ≈ {r.vessel_length_m:.0f} m<br>"
             f"Vitesse ≈ {r.vessel_speed_k:.1f} nœuds<br>"
             f"Probabilité pêche : {r.is_fishing_vessel:.0%}<br>"
             f"Confiance : {r.score:.2f}<br>{r.lat:.4f}, {r.lon:.4f}")
    folium.CircleMarker([r.lat, r.lon], radius=4, color="#0a84ff",
                        fill=True, fill_opacity=0.9, popup=popup).add_to(fmap)
fmap.save(os.path.join(OUT_ROOT, "carte_detections.html"))
fmap

## Et ensuite

Ce notebook répond à la première question : **où sont les navires ?**

Le notebook 02 répondra à la vraie question de VigieMer : **lesquels sont invisibles ?** On croisera chaque détection avec les positions AIS de Global Fishing Watch au même moment. Un navire vu par le radar, mais absent de l'AIS, devient une alerte « dark vessel ».

Fichiers produits dans `/kaggle/working/vigiemer_outputs/` : `detections.csv` par scène, `overview.png`, `crops.png` et `carte_detections.html`.